# Laporan Praktikum Association Rule Mining — Tugas 1 sampai 4
**Metode:** Apriori dan FP-Growth  
**Tempat pengerjaan:** Google Colab atau Jupyter Notebook

Pada praktikum ini saya menganalisis pola barang yang sering dibeli secara bersamaan. Untuk **Tugas 1 sampai 3**, saya memakai data transaksi toko kelontong dari file `transactions_arm_praktikum.csv` yang diberikan. File `transactions_arm_praktikum_onehot.csv` dipakai untuk mengecek hasil encoding. Untuk **Tugas 4**, saya tetap menggunakan studi kasus Pet Shop seperti pada jawaban sebelumnya; kode dan pembahasannya tidak diubah.

Agar notebook dapat dijalankan, simpan ketiga file CSV pada folder yang sama dengan notebook, atau unggah semuanya melalui menu **Files** di Google Colab. Jalankan sel secara berurutan dari atas ke bawah.


## Persiapan data

Langkah pertama adalah membaca transaksi toko kelontong. Setiap baris pada file transaksi berisi satu struk belanja. Karena Apriori dan FP-Growth membutuhkan data berbentuk True/False, saya mengubah daftar belanja menjadi tabel one-hot. Saya juga membandingkannya dengan file one-hot yang sudah disediakan supaya tidak ada item yang tertukar.


In [ ]:
%pip -q install mlxtend pandas numpy matplotlib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import time
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, fpgrowth, association_rules
from IPython.display import display

def load_transactions(path):
    raw = pd.read_csv(path, encoding='utf-8-sig')
    # File praktikum memakai TID; file Pet Shop memakai ID_Transaksi.
    if 'Items' not in raw.columns:
        raise ValueError('File CSV harus mempunyai kolom Items.')
    tx = [[item.strip() for item in str(items).split(',') if item.strip()]
          for items in raw['Items'].fillna('')]
    if not all(tx):
        raise ValueError('Ada transaksi kosong. Periksa kembali file CSV.')
    encoder = TransactionEncoder()
    matrix = pd.DataFrame(encoder.fit(tx).transform(tx), columns=encoder.columns_)
    return raw, tx, matrix

def make_rules(fi, threshold=0.5):
    if fi.empty or not any(len(items) > 1 for items in fi['itemsets']):
        return pd.DataFrame()
    return (association_rules(fi, metric='confidence', min_threshold=threshold)
            .sort_values('lift', ascending=False).reset_index(drop=True))

def pretty_rules(r):
    if r.empty:
        return pd.DataFrame()
    cols = ['antecedents', 'consequents', 'support', 'confidence',
            'lift', 'leverage', 'conviction']
    result = r[cols].copy()
    for col in ['antecedents', 'consequents']:
        result[col] = result[col].apply(lambda items: ', '.join(sorted(items)))
    return result.round(4)

raw, transactions, df = load_transactions('transactions_arm_praktikum.csv')
onehot_asli = pd.read_csv('transactions_arm_praktikum_onehot.csv', encoding='utf-8-sig')
# Samakan urutan kolom dan tipe data sebelum membandingkan.
onehot_asli = onehot_asli[df.columns].replace({'True': True, 'False': False}).astype(bool)
assert df.reset_index(drop=True).equals(onehot_asli.reset_index(drop=True)), (
    'Hasil encoding berbeda dari file one-hot yang diberikan!')
print('Data transaksi:', len(transactions), 'transaksi')
print('Jumlah jenis barang:', df.shape[1])
print('Pemeriksaan one-hot: sesuai dengan file yang diunggah')
display(raw.head(), df.head())
display(df.sum().sort_values(ascending=False).rename('Frekuensi').to_frame())


## Tugas 1 — Pengaruh nilai minimum support dan confidence

### 1.1 Membandingkan minimum support 0,10 dan 0,15

Saya mencoba dua nilai minimum support pada data yang sama. Nilai 0,10 berarti kombinasi barang harus muncul dalam setidaknya 10% transaksi, sedangkan 0,15 berarti minimal 15%. Confidence tetap saya tetapkan sebesar 0,50 supaya perbandingannya adil.


In [ ]:
fi10 = apriori(df,min_support=0.10,use_colnames=True)
fi15 = apriori(df,min_support=0.15,use_colnames=True)
r10 = make_rules(fi10,0.5)
r15 = make_rules(fi15,0.5)
comparison = pd.DataFrame({'min_support':[0.10,0.15], 'Jumlah itemset':[len(fi10),len(fi15)],'Jumlah aturan':[len(r10),len(r15)]})
display(comparison)
print('Perubahan itemset:',len(fi10)-len(fi15),'| Perubahan aturan:',len(r10)-len(r15))

### 1.2 Menaikkan minimum confidence menjadi 0,70

Selanjutnya saya mempertahankan minimum support 0,15, lalu menaikkan minimum confidence dari 0,50 menjadi 0,70. Tujuannya untuk melihat aturan mana yang masih cukup konsisten dan aturan mana yang tidak lagi memenuhi syarat.


In [ ]:
r70 = make_rules(fi15,0.7)
failed = r15[r15['confidence'] < 0.7]
print('Aturan awal:',len(r15),'| Lolos:',len(r70),'| Gugur:',len(failed))
print('Aturan yang gugur:')
display(pretty_rules(failed))
print('Aturan yang lolos:')
display(pretty_rules(r70))

### 1.3 Grafik perubahan jumlah aturan

Grafik berikut memperlihatkan perubahan jumlah aturan saat confidence dinaikkan secara bertahap. Minimum support tetap 0,15, sehingga satu-satunya nilai yang berubah adalah confidence.


In [ ]:
thresholds=[0.5,0.6,0.7,0.8,0.9,1.0]
counts=[len(make_rules(fi15,c)) for c in thresholds]
fig,ax=plt.subplots(figsize=(8,4))
ax.plot(thresholds,counts,marker='o');ax.set(xlabel='min_confidence',ylabel='Jumlah aturan',title='Tugas 1: Pengaruh min_confidence');ax.grid(alpha=.3)
for x,y in zip(thresholds,counts):ax.annotate(str(y),(x,y),xytext=(0,7),textcoords='offset points',ha='center')
plt.tight_layout();plt.show()
display(pd.DataFrame({'min_confidence':thresholds,'jumlah_aturan':counts}))

### 1.4 Pembahasan

Dari percobaan ini terlihat bahwa batas yang dipilih memengaruhi banyaknya pola yang bisa ditemukan. Saat minimum support diturunkan, kombinasi yang lebih jarang muncul masih bisa ikut dianalisis. Sebaliknya, ketika confidence dinaikkan, aturan yang tidak cukup konsisten akan tersaring.

Menurut saya, batas yang terlalu rendah memang menarik karena menghasilkan lebih banyak pilihan, tetapi belum tentu semuanya berguna untuk toko. Batas yang terlalu tinggi juga kurang ideal karena pola pembelian barang yang tidak terlalu populer bisa terlewat. Karena itu, saya tidak hanya melihat jumlah aturan, tetapi juga nilai support dan lift sebelum menyimpulkan bahwa suatu pola layak digunakan.


## Tugas 2 — Menghitung leverage dan conviction

Saya memilih dua aturan yang belum menjadi contoh utama pada bagian interpretasi bisnis modul, yaitu **Mie Instan → Telur** dan **Roti → Susu**. Keduanya saya hitung langsung dari transaksi agar nilai leverage dan conviction dapat dibandingkan dengan lift.

Rumus yang digunakan:
- **Leverage** = Support(X dan Y) − Support(X) × Support(Y).
- **Conviction** = (1 − Support(Y)) / (1 − Confidence(X → Y)).

Leverage positif berarti kedua barang muncul bersama lebih sering daripada yang diperkirakan jika pembeliannya saling bebas. Lift di atas 1 juga menunjukkan hubungan positif. Sementara itu, conviction membantu melihat seberapa kuat aturan jika dilihat dari kemungkinan terjadinya pengecualian.


In [ ]:
def direct_metrics(tx, X, Y):
    n = len(tx)
    X, Y = set(X), set(Y)
    sx = sum(X.issubset(set(t)) for t in tx) / n
    sy = sum(Y.issubset(set(t)) for t in tx) / n
    sxy = sum((X | Y).issubset(set(t)) for t in tx) / n
    confidence = sxy / sx if sx else float('nan')
    lift = confidence / sy if sy else float('nan')
    leverage = sxy - sx * sy
    conviction = ((1 - sy) / (1 - confidence)
                  if confidence < 1 else float('inf'))
    return {'Aturan': f"{', '.join(sorted(X))} → {', '.join(sorted(Y))}",
            'support': sxy, 'confidence': confidence,
            'lift': lift, 'leverage': leverage, 'conviction': conviction}

selected = pd.DataFrame([
    direct_metrics(transactions, ['Mie Instan'], ['Telur']),
    direct_metrics(transactions, ['Roti'], ['Susu'])
])
display(selected.round(4))


### Interpretasi bisnis

**Mie Instan → Telur.** Jika pembeli mi instan juga cukup sering membeli telur, toko dapat mencoba menempatkan kedua produk di area yang mudah dijangkau. Pilihan lainnya adalah menawarkan paket mi instan dan telur dengan harga yang menarik. Sebelum diterapkan, saya akan melihat apakah nilai lift lebih dari 1 dan apakah support-nya cukup besar.

**Roti → Susu.** Hubungan ini dapat dimanfaatkan untuk promosi sarapan, misalnya potongan harga susu ketika pelanggan membeli roti. Namun, promosi sebaiknya diuji terlebih dahulu karena aturan asosiasi hanya menunjukkan kebiasaan membeli bersama, bukan membuktikan bahwa pembelian roti menyebabkan pembelian susu.

Dari ketiga ukuran, lift lebih mudah digunakan untuk melihat kekuatan hubungan relatif, leverage menunjukkan selisih terhadap kemunculan yang diharapkan, sedangkan conviction melihat konsistensi arah aturan. Ketiganya saling melengkapi.


## Tugas 3 — Membandingkan Apriori dan FP-Growth

Pada tugas ini saya menjalankan kedua algoritma dengan minimum support **0,05** dan membandingkannya dengan **0,15**. Saya mencatat jumlah itemset dan waktu prosesnya. Pengukuran dilakukan beberapa kali lalu diambil nilai median supaya hasilnya tidak terlalu dipengaruhi satu kali proses yang kebetulan lebih lambat.


In [ ]:
def benchmark(matrix,support,repeats=15):
    records=[]
    for method,func in [('Apriori',apriori),('FP-Growth',fpgrowth)]:
        durations=[];result=None
        for _ in range(repeats):
            t0=time.perf_counter();result=func(matrix,min_support=support,use_colnames=True)
            durations.append((time.perf_counter()-t0)*1000)
        records.append({'min_support':support,'Algoritma':method,'Jumlah itemset':len(result),'Median waktu (ms)':np.median(durations),'Itemsets':set(map(frozenset,result.itemsets))})
    assert records[0]['Itemsets']==records[1]['Itemsets'],'Hasil kedua algoritma tidak identik!'
    return pd.DataFrame([{k:v for k,v in r.items() if k!='Itemsets'} for r in records])
bench=pd.concat([benchmark(df,0.05),benchmark(df,0.15)],ignore_index=True)
display(bench.round(4))
print('Acuan waktu dari modul untuk dataset asli pada support 0.15: Apriori 10.42 ms, FP-Growth 2.95 ms; tidak dapat dibandingkan langsung karena dataset/perangkat berbeda.')
for s in [0.05,0.15]:
    part=bench[bench.min_support==s].set_index('Algoritma')
    a=part.loc['Apriori','Median waktu (ms)'];f=part.loc['FP-Growth','Median waktu (ms)']
    print(f'support={s:.2f}: FP-Growth {a/f:.2f}× lebih cepat daripada Apriori' if f<a else f'support={s:.2f}: Apriori {f/a:.2f}× lebih cepat daripada FP-Growth')
fig,ax=plt.subplots(figsize=(8,4))
for name,g in bench.groupby('Algoritma'):
    ax.plot(g.min_support,g['Median waktu (ms)'],marker='o',label=name)
ax.set(xlabel='min_support',ylabel='Median waktu eksekusi (ms)',title='Tugas 3: Perbandingan runtime');ax.legend();ax.grid(alpha=.3);plt.tight_layout();plt.show()

### Pembahasan hasil perbandingan

Apriori mencari frequent itemset dengan membentuk dan memeriksa kandidat secara bertahap. FP-Growth menggunakan struktur FP-Tree sehingga tidak perlu membuat kandidat dengan cara yang sama. Secara teori, FP-Growth sering lebih menguntungkan saat kombinasi yang diperiksa semakin banyak.

Walaupun demikian, data praktikum hanya berisi 60 transaksi. Pada data sekecil ini, perbedaan waktu dapat dipengaruhi oleh kondisi komputer dan proses lain yang sedang berjalan. Oleh karena itu, jawaban apakah FP-Growth lebih cepat **harus mengikuti angka median yang muncul pada tabel**, bukan hanya berdasarkan teori. Kedua algoritma juga diperiksa agar menghasilkan frequent itemset yang sama pada batas support yang sama.


## Tugas 4 — Studi kasus mandiri: Pet Shop
Dataset sintetis 60 transaksi, 10 jenis produk. Parameter dipilih `min_support=0.15` (setidaknya 9 dari 60 transaksi) dan `min_confidence=0.50` (sedikitnya separuh transaksi antecedent mengandung consequent).

In [ ]:
pet_raw,pet_tx,pet_df=load_transactions('Dataset_Tugas_4_Pet_Shop.csv')
print('Jumlah transaksi:',len(pet_tx),'| Jumlah jenis produk:',pet_df.shape[1]);display(pet_raw.head());display(pet_df.head())
pet_fi=apriori(pet_df,min_support=0.15,use_colnames=True).sort_values('support',ascending=False)
pet_rules=make_rules(pet_fi,0.50)
print('Frequent itemset:',len(pet_fi),'| Aturan:',len(pet_rules))
display(pet_fi.head(15),pretty_rules(pet_rules).head(15))

**Visualisasi 1:** Scatter plot Support vs Confidence, warna dan ukuran gelembung merepresentasikan Lift.

In [ ]:
if len(pet_rules):
    fig,ax=plt.subplots(figsize=(8,5))
    dots=ax.scatter(pet_rules.support,pet_rules.confidence,c=pet_rules.lift,s=pet_rules.lift*90,cmap='viridis',alpha=.8,edgecolor='black',linewidth=.3)
    fig.colorbar(dots,ax=ax,label='Lift');ax.set(xlabel='Support',ylabel='Confidence',title='Pet Shop — Support vs Confidence');ax.grid(alpha=.2)
    plt.tight_layout();plt.show()

**Visualisasi 2:** Delapan aturan dengan Lift tertinggi.

In [ ]:
if len(pet_rules):
    top=pretty_rules(pet_rules).head(8).copy()
    top['Aturan']=top['antecedents']+' → '+top['consequents']
    fig,ax=plt.subplots(figsize=(10,5))
    ax.barh(top['Aturan'][::-1],top['lift'][::-1]);ax.set(xlabel='Lift',title='Pet Shop — 8 Aturan Lift Tertinggi')
    plt.tight_layout();plt.show()
    display(top[['Aturan','support','confidence','lift']])

**Rekomendasi bisnis (minimal 3):**
1. **Bundling perlengkapan mandi:** Jika aturan `Shampo Hewan + Mainan Hewan → Sisir Hewan` muncul dengan Lift tinggi, buat paket grooming berisi ketiganya dan uji diskon kecil.
2. **Cross-selling kebutuhan kucing:** Jika aturan `Makanan Kucing → Pasir Kucing` memiliki Lift > 1, tampilkan rekomendasi pasir kucing di kasir/checkout untuk pembeli makanan kucing.
3. **Penataan rak kebutuhan anjing:** Jika aturan `Makanan Anjing → Snack Anjing` memiliki Lift > 1, tempatkan snack anjing berdekatan dengan makanan anjing.

**Validasi:** Rekomendasi di atas merupakan hipotesis berbasis kelompok produk; **pastikan** aturan tersebut benar-benar ada dan memiliki Lift > 1 pada tabel output. Jika tidak, gunakan tiga aturan dengan Lift tertinggi yang benar-benar muncul di bawah ini.

In [ ]:
if len(pet_rules):
    print('Tiga aturan Lift tertinggi yang TERBUKTI pada dataset:')
    display(pretty_rules(pet_rules).head(3))
    pretty_rules(pet_rules).to_csv('Hasil_Aturan_Pet_Shop.csv',index=False)
    print('Disimpan: Hasil_Aturan_Pet_Shop.csv')

## Kesimpulan

Setelah mengerjakan praktikum ini, saya memahami bahwa association rule mining dapat membantu menemukan barang yang sering dibeli secara bersamaan. Minimum support menentukan seberapa sering suatu kombinasi harus muncul, sedangkan minimum confidence menunjukkan seberapa konsisten suatu aturan. Lift, leverage, dan conviction membantu menilai hubungan tersebut dari sudut pandang yang berbeda.

Pada Tugas 1–3 saya menggunakan dataset transaksi kelontong yang diberikan, sedangkan pada Tugas 4 saya menggunakan dataset Pet Shop sintetis. Hasil analisis dapat menjadi masukan untuk menyusun paket promo, rekomendasi produk, atau penataan rak. Namun, keputusan bisnis tetap perlu diuji karena pola asosiasi tidak otomatis menunjukkan hubungan sebab-akibat.

**Referensi:** Modul Praktikum Data Mining — Association Rule Mining (Universitas Negeri Padang, 2026); Agrawal & Srikant (1994); Han, Pei, & Yin (2000); Raschka (2018).
